# Transfer Learning untuk Klasifikasi Sampah

Notebook ini merupakan implementasi Google Colab dari project **Transfer Learning Waste Classification** menggunakan **ResNet-18**.

Kelas yang digunakan:
- Plastic
- Paper
- Metal
- Glass

Empat skenario eksperimen:
1. Feature — ImageNet, hanya `fc`, learning rate `1e-3`
2. Partial — ImageNet, `layer4 + fc`, learning rate `1e-4`
3. Partial — ImageNet, `layer4 + fc`, learning rate `1e-3`
4. Scratch — random initialization, seluruh layer dilatih, learning rate `1e-3`

Konfigurasi utama:
- Arsitektur: ResNet-18
- Epoch: 20
- Batch size: 16
- Input: 224 × 224 pixel
- Optimizer: Adam
- Loss: CrossEntropyLoss
- Split: 70% train, 15% validation, 15% test
- Random seed: 42


## 1. Setup Google Colab dan Clone Repository

Repository GitHub yang digunakan sebagai sumber dataset dan kode project.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

REPO_URL = "https://github.com/KakiJawa/transfer_learning_RET503.git"
REPO_DIR = Path("/content/transfer_learning_RET503")

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    ["git", "clone", "-q", REPO_URL, str(REPO_DIR)],
    check=True
)

os.chdir(REPO_DIR)

print("Repository berhasil di-clone.")
print("Project directory:", Path.cwd())

## 2. Install Library

PyTorch dan torchvision menggunakan versi yang sudah tersedia pada Google Colab. Library tambahan di-install jika diperlukan.

In [ ]:
!pip -q install matplotlib scikit-learn pandas Pillow

## 3. Import Library dan Pemeriksaan GPU

In [ ]:
import os
import random
import time
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import resnet18, ResNet18_Weights

print("PyTorch version :", torch.__version__)
print("Torchvision     :", __import__('torchvision').__version__)
print("CUDA available  :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU             :", torch.cuda.get_device_name(0))
    print("CUDA device     :", torch.cuda.current_device())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device          :", device)

## 4. Konfigurasi Project

Seed dan konfigurasi dibuat sama dengan project lokal agar pembagian dataset dan eksperimen dapat direproduksi.

In [ ]:
SEED = 42
NUM_CLASSES = 4
BATCH_SIZE = 16
IMAGE_SIZE = 224
EPOCHS = 20

CLASS_NAMES = ["glass", "metal", "paper", "plastic"]

RAW_DIR = Path("dataset_raw")
SPLIT_DIR = Path("dataset_split")
RESULTS_DIR = Path("results_colab")
MODELS_DIR = Path("models_colab")

RESULTS_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

print("Seed       :", SEED)
print("Batch size :", BATCH_SIZE)
print("Image size :", IMAGE_SIZE)
print("Epochs     :", EPOCHS)

## 5. Pemeriksaan Dataset Raw

Dataset raw berasal dari TrashNet dan digunakan untuk empat kelas sampah.

In [ ]:
valid_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

total_images = 0
corrupt_images = []

for class_name in CLASS_NAMES:
    class_dir = RAW_DIR / class_name
    files = [p for p in class_dir.rglob("*") if p.suffix.lower() in valid_extensions]
    print(f"{class_name:8s}: {len(files)} gambar")
    total_images += len(files)

    for file_path in files:
        try:
            with Image.open(file_path) as img:
                img.verify()
        except Exception:
            corrupt_images.append(str(file_path))

print("\nTotal gambar :", total_images)
print("Gambar rusak:", len(corrupt_images))

if corrupt_images:
    print("File rusak:")
    for item in corrupt_images:
        print(item)
else:
    print("Semua gambar dapat dibaca dengan baik.")

## 6. Membaca Metadata Dataset

In [ ]:
metadata_path = Path("metadata.csv")

if metadata_path.exists():
    metadata_df = pd.read_csv(metadata_path)
    display(metadata_df.head())
    print("Jumlah data metadata:", len(metadata_df))
else:
    print("metadata.csv tidak ditemukan.")

## 7. Membuat Dataset Split

Pembagian dataset mengikuti project lokal:

- Train = 70%
- Validation = 15%
- Test = 15%
- Seed = 42

`dataset_split` dibuat ulang dari `dataset_raw` sehingga notebook tidak bergantung pada folder split yang di-ignore oleh Git.

In [ ]:
if SPLIT_DIR.exists():
    shutil.rmtree(SPLIT_DIR)

for split in ["train", "val", "test"]:
    for class_name in CLASS_NAMES:
        (SPLIT_DIR / split / class_name).mkdir(parents=True, exist_ok=True)

for class_name in CLASS_NAMES:
    source_dir = RAW_DIR / class_name
    files = sorted([
        p for p in source_dir.rglob("*")
        if p.suffix.lower() in valid_extensions
    ])

    train_files, temp_files = train_test_split(
        files,
        test_size=0.30,
        random_state=SEED
    )

    val_files, test_files = train_test_split(
        temp_files,
        test_size=0.50,
        random_state=SEED
    )

    split_files = {
        "train": train_files,
        "val": val_files,
        "test": test_files,
    }

    for split_name, selected_files in split_files.items():
        for source_file in selected_files:
            destination = SPLIT_DIR / split_name / class_name / source_file.name
            shutil.copy2(source_file, destination)

    print(
        f"{class_name:8s} | train: {len(train_files):3d} | "
        f"val: {len(val_files):3d} | test: {len(test_files):3d}"
    )

## 8. Transformasi dan DataLoader

Transformasi train menggunakan augmentasi sederhana. Validation dan test tidak menggunakan augmentasi.

In [ ]:
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

train_dataset = datasets.ImageFolder(SPLIT_DIR / "train", transform=train_transform)
val_dataset = datasets.ImageFolder(SPLIT_DIR / "val", transform=eval_transform)
test_dataset = datasets.ImageFolder(SPLIT_DIR / "test", transform=eval_transform)

NUM_WORKERS = 2 if torch.cuda.is_available() else 0

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print("Classes:", train_dataset.classes)
print("Class mapping:", train_dataset.class_to_idx)
print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))

images, labels = next(iter(train_loader))
print("Batch image shape:", images.shape)
print("Batch label shape:", labels.shape)

## 9. Contoh Dataset

In [ ]:
display_dataset = datasets.ImageFolder(SPLIT_DIR / "train")
sample_indices = np.linspace(
    0,
    len(display_dataset) - 1,
    8,
    dtype=int
)

plt.figure(figsize=(14, 7))

for i, idx in enumerate(sample_indices):
    image, label = display_dataset[idx]
    plt.subplot(2, 4, i + 1)
    plt.imshow(image)
    plt.title(display_dataset.classes[label])
    plt.axis("off")

plt.tight_layout()
plt.show()

## 10. Arsitektur ResNet-18 dan Mode Eksperimen

Implementasi dibuat mengikuti project lokal.

**Feature:** seluruh backbone dibekukan dan hanya `fc` yang dilatih.

**Partial:** seluruh backbone awal dibekukan, sedangkan `layer4` dan `fc` dilatih.

**Scratch:** model dibuat tanpa pretrained weights dan seluruh parameter dilatih.

In [ ]:
def create_model(mode):
    if mode == "feature":
        model = resnet18(weights=ResNet18_Weights.DEFAULT)

        for param in model.parameters():
            param.requires_grad = False

        model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

        for param in model.fc.parameters():
            param.requires_grad = True

    elif mode == "partial":
        model = resnet18(weights=ResNet18_Weights.DEFAULT)

        for param in model.parameters():
            param.requires_grad = False

        for param in model.layer4.parameters():
            param.requires_grad = True

        model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

        for param in model.fc.parameters():
            param.requires_grad = True

    elif mode == "scratch":
        model = resnet18(weights=None)
        model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

        for param in model.parameters():
            param.requires_grad = True

    else:
        raise ValueError("Mode harus feature, partial, atau scratch")

    return model


def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(
        p.numel() for p in model.parameters() if p.requires_grad
    )
    return total, trainable


for mode in ["feature", "partial", "scratch"]:
    model = create_model(mode)
    total, trainable = count_parameters(model)
    print(f"{mode:8s} | total: {total:,} | trainable: {trainable:,}")

## 11. Fungsi Training dan Validation

In [ ]:
criterion = nn.CrossEntropyLoss()


def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.set_grad_enabled(is_train):
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            if is_train:
                optimizer.zero_grad(set_to_none=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            if is_train:
                loss.backward()
                optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc


def train_model(mode, lr, epochs=20):
    set_seed(SEED)

    model = create_model(mode).to(device)
    trainable_params = [
        p for p in model.parameters() if p.requires_grad
    ]

    optimizer = torch.optim.Adam(trainable_params, lr=lr)

    history = {
        "epoch": [],
        "train_loss": [],
        "train_accuracy": [],
        "val_loss": [],
        "val_accuracy": [],
    }

    best_val_acc = -1.0
    best_state = None

    start_time = time.perf_counter()

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = run_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
        )

        val_loss, val_acc = run_epoch(
            model,
            val_loader,
            criterion,
        )

        history["epoch"].append(epoch)
        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }

        print(
            f"Epoch {epoch:02d}/{epochs} | "
            f"Train Loss {train_loss:.4f} | "
            f"Train Acc {train_acc * 100:.2f}% | "
            f"Val Loss {val_loss:.4f} | "
            f"Val Acc {val_acc * 100:.2f}%"
        )

    training_time = time.perf_counter() - start_time

    model.load_state_dict(best_state)
    model = model.to(device)

    return model, history, best_val_acc, training_time

## 12. Menjalankan 4 Eksperimen

Setiap eksperimen menggunakan 20 epoch dan checkpoint terbaik berdasarkan validation accuracy.

In [ ]:
experiments = [
    {
        "name": "feature_lr1e-3",
        "mode": "feature",
        "lr": 1e-3,
    },
    {
        "name": "partial_lr1e-4",
        "mode": "partial",
        "lr": 1e-4,
    },
    {
        "name": "partial_lr1e-3",
        "mode": "partial",
        "lr": 1e-3,
    },
    {
        "name": "scratch_lr1e-3",
        "mode": "scratch",
        "lr": 1e-3,
    },
]

experiment_results = {}

for exp in experiments:
    print("=" * 70)
    print(f"EXPERIMENT: {exp['name']}")
    print(f"Mode: {exp['mode']} | Learning Rate: {exp['lr']}")
    print("=" * 70)

    model, history, best_val_acc, training_time = train_model(
        exp["mode"],
        exp["lr"],
        EPOCHS,
    )

    history_df = pd.DataFrame(history)
    history_df.to_csv(
        RESULTS_DIR / f"{exp['name']}_history.csv",
        index=False,
    )

    model_path = MODELS_DIR / f"{exp['name']}_best.pth"
    torch.save(model.state_dict(), model_path)

    experiment_results[exp["name"]] = {
        "name": exp["name"],
        "mode": exp["mode"],
        "lr": exp["lr"],
        "model": model,
        "history": history_df,
        "best_val_accuracy": best_val_acc,
        "training_time": training_time,
        "model_path": str(model_path),
    }

    print(f"Best Val Accuracy: {best_val_acc * 100:.2f}%")
    print(f"Training Time: {training_time:.2f} s")


## 13. Grafik Accuracy dan Loss

In [ ]:
for name, result in experiment_results.items():
    history = result["history"]

    fig = plt.figure(figsize=(8, 5))
    plt.plot(
        history["epoch"],
        history["train_accuracy"] * 100,
        label="Train Accuracy",
    )
    plt.plot(
        history["epoch"],
        history["val_accuracy"] * 100,
        label="Validation Accuracy",
    )
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy (%)")
    plt.title(f"Accuracy - {name}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f"{name}_accuracy.png", dpi=150)
    plt.show()
    plt.close(fig)

    fig = plt.figure(figsize=(8, 5))
    plt.plot(
        history["epoch"],
        history["train_loss"],
        label="Train Loss",
    )
    plt.plot(
        history["epoch"],
        history["val_loss"],
        label="Validation Loss",
    )
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"Loss - {name}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f"{name}_loss.png", dpi=150)
    plt.show()
    plt.close(fig)

## 14. Evaluasi Test Set

Model yang digunakan adalah checkpoint dengan validation accuracy terbaik dari masing-masing eksperimen.

In [ ]:
def evaluate_model(model, loader):
    model.eval()
    y_true = []
    y_pred = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            outputs = model(images)
            predictions = outputs.argmax(dim=1).cpu().numpy()

            y_pred.extend(predictions.tolist())
            y_true.extend(labels.numpy().tolist())

    accuracy = accuracy_score(y_true, y_pred)
    return accuracy, np.array(y_true), np.array(y_pred)


evaluation_rows = []

for name, result in experiment_results.items():
    accuracy, y_true, y_pred = evaluate_model(result["model"], test_loader)

    result["test_accuracy"] = accuracy
    result["y_true"] = y_true
    result["y_pred"] = y_pred

    evaluation_rows.append({
        "Experiment": name,
        "Mode": result["mode"],
        "Learning Rate": result["lr"],
        "Best Val Accuracy": result["best_val_accuracy"],
        "Test Accuracy": accuracy,
        "Training Time (s)": result["training_time"],
    })

comparison_df = pd.DataFrame(evaluation_rows)
comparison_df.to_csv(RESULTS_DIR / "comparison_summary.csv", index=False)

display(comparison_df.style.format({
    "Learning Rate": "{:.0e}",
    "Best Val Accuracy": "{:.2%}",
    "Test Accuracy": "{:.2%}",
    "Training Time (s)": "{:.2f}",
}))

## 15. Classification Report Per Kelas

In [ ]:
for name, result in experiment_results.items():
    report_dict = classification_report(
        result["y_true"],
        result["y_pred"],
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0,
    )

    report_df = pd.DataFrame(report_dict).transpose()
    report_df.to_csv(
        RESULTS_DIR / f"{name}_classification_report.csv"
    )

    print("=" * 70)
    print(name)
    print("=" * 70)
    print(
        classification_report(
            result["y_true"],
            result["y_pred"],
            target_names=CLASS_NAMES,
            zero_division=0,
        )
    )

## 16. Confusion Matrix

In [ ]:
for name, result in experiment_results.items():
    cm = confusion_matrix(
        result["y_true"],
        result["y_pred"],
    )

    fig, ax = plt.subplots(figsize=(6, 6))
    display_cm = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=CLASS_NAMES,
    )
    display_cm.plot(ax=ax, values_format="d", cmap="Blues")
    ax.set_title(f"Confusion Matrix - {name}")
    plt.tight_layout()
    plt.savefig(
        RESULTS_DIR / f"{name}_confusion_matrix.png",
        dpi=150,
    )
    plt.show()
    plt.close(fig)

## 17. Perbandingan Hasil Eksperimen

In [ ]:
fig = plt.figure(figsize=(9, 5))
plt.bar(
    comparison_df["Experiment"],
    comparison_df["Test Accuracy"] * 100,
)
plt.ylabel("Test Accuracy (%)")
plt.xlabel("Experiment")
plt.title("Comparison of Test Accuracy")
plt.xticks(rotation=20)
plt.grid(axis="y")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "comparison_accuracy.png", dpi=150)
plt.show()
plt.close(fig)


fig = plt.figure(figsize=(9, 5))
plt.bar(
    comparison_df["Experiment"],
    comparison_df["Training Time (s)"],
)
plt.ylabel("Training Time (seconds)")
plt.xlabel("Experiment")
plt.title("Comparison of Training Time")
plt.xticks(rotation=20)
plt.grid(axis="y")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "comparison_training_time.png", dpi=150)
plt.show()
plt.close(fig)

## 18. Eksperimen dengan Test Accuracy Tertinggi

Bagian ini hanya mengidentifikasi eksperimen dengan **test accuracy tertinggi pada empat eksperimen yang dijalankan**, bukan menyatakan bahwa model tersebut merupakan model terbaik secara umum.

In [ ]:
best_test_idx = comparison_df["Test Accuracy"].idxmax()
highest_test = comparison_df.loc[best_test_idx]

print("Eksperimen dengan test accuracy tertinggi pada percobaan ini:")
print("Experiment       :", highest_test["Experiment"])
print("Mode             :", highest_test["Mode"])
print("Learning Rate    :", f"{highest_test['Learning Rate']:.0e}")
print("Best Val Accuracy:", f"{highest_test['Best Val Accuracy']:.2%}")
print("Test Accuracy    :", f"{highest_test['Test Accuracy']:.2%}")

## 19. Pengukuran Latency Inference

Latency diukur menggunakan model dengan test accuracy tertinggi pada eksperimen ini. Pengukuran menggunakan 20 warm-up dan 100 inference terukur.

Hasil latency Colab tidak digunakan untuk menggantikan hasil latency pada laptop lokal karena perangkat kerasnya berbeda.

In [ ]:
selected_name = highest_test["Experiment"]
selected_model = experiment_results[selected_name]["model"]
selected_model.eval()

sample_image, _ = test_dataset[0]
input_tensor = sample_image.unsqueeze(0).to(device)

WARMUP = 20
RUNS = 100

with torch.no_grad():
    for _ in range(WARMUP):
        _ = selected_model(input_tensor)

if torch.cuda.is_available():
    torch.cuda.synchronize()

times = []

with torch.no_grad():
    for _ in range(RUNS):
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        start = time.perf_counter()
        _ = selected_model(input_tensor)

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        end = time.perf_counter()
        times.append((end - start) * 1000)

avg_latency = np.mean(times)
min_latency = np.min(times)
max_latency = np.max(times)
estimated_fps = 1000 / avg_latency

print(f"Model             : {selected_name}")
print(f"Average latency   : {avg_latency:.3f} ms")
print(f"Minimum latency   : {min_latency:.3f} ms")
print(f"Maximum latency   : {max_latency:.3f} ms")
print(f"Estimated FPS     : {estimated_fps:.2f}")
print("\nCatatan: latency ini hanya inference model, bukan end-to-end camera + preprocessing + inference.")

In [ ]:
latency_df = pd.DataFrame([{
    "Experiment": selected_name,
    "Average Latency (ms)": avg_latency,
    "Minimum Latency (ms)": min_latency,
    "Maximum Latency (ms)": max_latency,
    "Estimated FPS": estimated_fps,
}])

latency_df.to_csv(RESULTS_DIR / "latency.csv", index=False)
display(latency_df)

## 20. Analisis Singkat

Beberapa hal yang perlu diperhatikan dari hasil eksperimen:

1. **Feature extraction** hanya melatih fully connected layer sehingga jumlah parameter yang diperbarui jauh lebih sedikit.
2. **Partial fine-tuning** memungkinkan layer akhir ResNet-18 beradaptasi terhadap karakteristik dataset sampah.
3. Learning rate yang berbeda pada partial fine-tuning dapat menghasilkan perilaku training yang berbeda.
4. **Scratch training** tidak menggunakan bobot ImageNet sehingga seluruh fitur harus dipelajari dari dataset yang relatif kecil.
5. Perbedaan antara train accuracy dan validation accuracy dapat digunakan untuk mengamati indikasi overfitting.
6. Test set hanya terdiri dari 60 gambar. Dengan demikian, satu gambar mewakili sekitar **1,67 percentage point** pada test accuracy.
7. Hasil training dan latency di Google Colab dapat berbeda dari hasil pada laptop lokal karena GPU, CPU, driver, dan lingkungan software berbeda.
8. FPS yang dihitung dari latency merupakan estimasi kemampuan inference model dan bukan throughput keseluruhan sistem kamera.

## 21. File Output

Notebook menyimpan hasil Colab pada folder terpisah agar tidak menimpa hasil eksperimen lokal:
- `results_colab/`
- `models_colab/`

Contoh file:
- history CSV
- classification report
- confusion matrix
- comparison summary
- comparison plots
- latency result
- model checkpoint

In [ ]:
print("Isi results_colab:")
for path in sorted(RESULTS_DIR.iterdir()):
    print("-", path.name)

print("\nIsi models_colab:")
for path in sorted(MODELS_DIR.iterdir()):
    print("-", path.name)

## 22. Repository GitHub

Project repository:

**https://github.com/KakiJawa/transfer_learning_RET503**

Notebook ini dibuat sebagai reproduksi eksperimen project menggunakan Google Colab.